# Session 9: Tuning the physics loss weight λ in PINNs

Every PINN training objective has the form:

$$\mathcal{L}(\theta) = \mathcal{L}_{\text{data}} + \lambda\,\mathcal{L}_{\text{physics}}$$

The scalar $\lambda > 0$ controls how strongly the PDE residual is enforced relative to the data fit. Getting it right matters:

- **Too small**: the physics constraint has negligible effect — the physics-informed neural network (PINN) behaves like a standard neural network.
- **Too large**: the ordinary differential equation (ODE) residual swamps the data loss — the model satisfies the equation but may refuse to fit the observations.
- **About right**: physics acts as a regulariser, enabling accurate extrapolation far beyond the training window.

This notebook investigates the effect of $\lambda$ systematically. We use the simple harmonic oscillator ODE as a test bed because it has an exact solution, making it easy to measure error honestly.

### Test problem

$$u''(x) + u(x) = 0, \quad x \in [0,\, 4\pi]$$

with exact solution $u(x) = \sin(x)$. We train on **5 noisy points in $[0, 2\pi]$** and measure extrapolation performance on $[2\pi, 4\pi]$ — a region the network never sees during training.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)

# ── Domain ────────────────────────────────────────────────────────────────────
TRAIN_LO, TRAIN_HI = 0.0, 2 * np.pi   # training window
EXTRAP_HI          = 4 * np.pi         # extrapolation up to here
N_TRAIN            = 5                 # noisy data points
NOISE              = 0.02              # measurement noise std
N_COLLOC           = 50               # collocation points (physics enforcement)

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 10000
LR     = 1e-3

## 2. Network, physics loss, and training function

In [ ]:
class MLP(nn.Module):
    """Small MLP: scalar x → scalar u(x). Three hidden layers of width 16."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.Tanh(),
            nn.Linear(16, 16), nn.Tanh(),
            nn.Linear(16, 16), nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


def physics_loss(model, x_col):
    """MSE of the ODE residual u'' + u = 0 at collocation points."""
    x_col = x_col.clone().requires_grad_(True)
    u     = model(x_col)
    du    = torch.autograd.grad(u,  x_col,
                                grad_outputs=torch.ones_like(u),
                                create_graph=True)[0]
    d2u   = torch.autograd.grad(du, x_col,
                                grad_outputs=torch.ones_like(du),
                                create_graph=True)[0]
    return torch.mean((d2u + u) ** 2)


def train(lam, x_train_t, y_train_t, x_col, seed=0):
    """Train one MLP with physics weight lam. Returns (model, loss_history)."""
    torch.manual_seed(seed)
    model   = MLP()
    opt     = optim.Adam(model.parameters(), lr=LR)
    mse_fn  = nn.MSELoss()
    history = []

    for _ in range(EPOCHS):
        opt.zero_grad()
        loss_data = mse_fn(model(x_train_t), y_train_t)
        loss_phys = physics_loss(model, x_col)
        loss      = loss_data + lam * loss_phys
        loss.backward()
        opt.step()
        history.append(loss.item())

    return model, history

## 3. Data and evaluation grid

In [ ]:
# Training data: 5 noisy observations in [0, 2π]
x_train = np.linspace(TRAIN_LO, TRAIN_HI, N_TRAIN).reshape(-1, 1)
y_train = np.sin(x_train) + np.random.normal(0, NOISE, x_train.shape)
x_train_t = torch.FloatTensor(x_train)
y_train_t = torch.FloatTensor(y_train)

# Collocation points: physics enforced across the training window
x_col = torch.FloatTensor(np.linspace(TRAIN_LO, TRAIN_HI, N_COLLOC).reshape(-1, 1))

# Evaluation grid spanning both training and extrapolation zones
x_interp = np.linspace(TRAIN_LO, TRAIN_HI, 200).reshape(-1, 1)
x_extrap  = np.linspace(TRAIN_HI, EXTRAP_HI, 200).reshape(-1, 1)
x_eval    = np.vstack([x_interp, x_extrap])
y_true    = np.sin(x_eval)
x_eval_t  = torch.FloatTensor(x_eval)

print(f"Training data:   {N_TRAIN} points in [0, 2π]")
print(f"Collocation pts: {N_COLLOC} in [0, 2π]")
print(f"Eval grid:       {len(x_eval)} points in [0, 4π]")

# Quick look at the data vs. exact solution
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(x_eval, y_true, 'k-', lw=2, label='Exact: sin(x)')
ax.scatter(x_train, y_train, s=60, c='green', zorder=5, label='Training data')
ax.axvspan(TRAIN_LO, TRAIN_HI, alpha=0.08, color='green', label='Training window')
ax.axvspan(TRAIN_HI, EXTRAP_HI, alpha=0.06, color='red',   label='Extrapolation zone')
ax.axvline(TRAIN_HI, color='gray', lw=1, ls=':')
ax.set_xlabel('x'); ax.set_ylabel('u(x)')
ax.set_title('Training setup: 5 noisy points, evaluate beyond 2π')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. The two extremes

Before sweeping λ systematically, it helps to see the two failure modes side by side:

| Setting | Behaviour |
|---|---|
| **λ = 0** (pure data) | Physics constraint is off entirely. The network fits the 5 noisy points but extrapolates randomly. |
| **λ = 10** (physics dominant) | The ODE residual completely swamps the data term. Physics is well satisfied but data fidelity may suffer. |
| **λ = 0.01** (balanced) | Both terms contribute. Data is fit and the ODE guides the prediction outside the training window. |

In [ ]:
print("Training λ = 0   (no physics)…")
model_none,  _ = train(lam=0.0,  x_train_t=x_train_t, y_train_t=y_train_t, x_col=x_col, seed=1)
print("Training λ = 0.01 (balanced)…")
model_bal,   _ = train(lam=0.01, x_train_t=x_train_t, y_train_t=y_train_t, x_col=x_col, seed=1)
print("Training λ = 10  (physics heavy)…")
model_heavy, _ = train(lam=10.0, x_train_t=x_train_t, y_train_t=y_train_t, x_col=x_col, seed=1)

with torch.no_grad():
    pred_none  = model_none(x_eval_t).numpy()
    pred_bal   = model_bal(x_eval_t).numpy()
    pred_heavy = model_heavy(x_eval_t).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
cases = [
    (pred_none,  r'$\lambda = 0$  (no physics)',     'tomato'),
    (pred_bal,   r'$\lambda = 0.01$  (balanced)',    'royalblue'),
    (pred_heavy, r'$\lambda = 10$  (physics heavy)', 'darkorange'),
]
for ax, (pred, title, col) in zip(axes, cases):
    ax.plot(x_eval, y_true, 'k-', lw=2, label='Exact sin(x)', alpha=0.8)
    ax.plot(x_eval, pred,   '-',  lw=2, color=col, label='Prediction')
    ax.scatter(x_train, y_train, s=60, c='green', zorder=5, label='Training data')
    ax.axvspan(TRAIN_LO, TRAIN_HI, alpha=0.08, color='green')
    ax.axvspan(TRAIN_HI, EXTRAP_HI, alpha=0.06, color='red')
    ax.axvline(TRAIN_HI, color='gray', lw=1, ls=':')
    ax.set_title(title, fontsize=12, color=col, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('u(x)')
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Training region (green) vs. extrapolation zone (red)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Systematic sweep over λ

We now train a separate model for each value of λ on a logarithmically-spaced grid and record three metrics:

| Metric | What it measures |
|---|---|
| **Interpolation error** | Mean absolute error inside the training window $[0, 2\pi]$ |
| **Extrapolation error** | Mean absolute error outside the training window $[2\pi, 4\pi]$ |
| **Physics residual** | Mean $|u'' + u|$ over the full domain |

A good λ minimises extrapolation error and physics residual without significantly worsening interpolation accuracy.

In [ ]:
lambdas = [0.0, 0.001, 0.01, 0.1, 0.7, 1.0]
results = {}

for lam in lambdas:
    print(f"Training λ = {lam}…")
    model, losses = train(lam, x_train_t, y_train_t, x_col, seed=0)

    with torch.no_grad():
        pred = model(x_eval_t).numpy()

    # Compute ODE residual for physics metric
    x_phys = x_eval_t.clone().requires_grad_(True)
    u   = model(x_phys)
    du  = torch.autograd.grad(u,  x_phys, grad_outputs=torch.ones_like(u),
                              create_graph=True)[0]
    d2u = torch.autograd.grad(du, x_phys, grad_outputs=torch.ones_like(du),
                              create_graph=True)[0]
    residual = (d2u + u).detach().numpy()

    results[lam] = {
        'pred':            pred,
        'error_interp':    float(np.mean(np.abs(pred[:200] - y_true[:200]))),
        'error_extrap':    float(np.mean(np.abs(pred[200:] - y_true[200:]))),
        'phys_residual':   float(np.mean(np.abs(residual))),
        'losses':          losses,
    }

print("\nSweep complete.")
print(f"{'λ':<8} {'Interp error':<16} {'Extrap error':<16} {'Physics residual'}")
print("-" * 58)
for lam, r in results.items():
    print(f"{lam:<8.4f} {r['error_interp']:<16.6f} {r['error_extrap']:<16.6f} {r['phys_residual']:.6f}")

## 6. Visualising the sweep

In [ ]:
# ── Panel 1: predictions for each λ ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(lambdas)))

for ax, (lam, col) in zip(axes.flatten(), zip(lambdas, colors)):
    pred = results[lam]['pred']
    ax.plot(x_eval, y_true, 'k-', lw=2, alpha=0.7, label='Exact')
    ax.plot(x_eval, pred,   '-',  lw=2, color=col,  label=f'λ={lam}')
    ax.scatter(x_train, y_train, s=40, c='green', zorder=5)
    ax.axvspan(TRAIN_LO, TRAIN_HI, alpha=0.07, color='green')
    ax.axvspan(TRAIN_HI, EXTRAP_HI, alpha=0.05, color='red')
    ax.axvline(TRAIN_HI, color='gray', lw=1, ls=':')
    ei = results[lam]['error_interp']
    ee = results[lam]['error_extrap']
    ax.set_title(f'λ = {lam}\ninterp={ei:.3f}  extrap={ee:.3f}', fontsize=10)
    ax.set_ylim(-3, 3)
    ax.set_xlabel('x'); ax.set_ylabel('u(x)')
    ax.grid(alpha=0.3)

plt.suptitle('Prediction quality for each λ (green = training, red = extrapolation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Panel 2: error and residual metrics vs λ ─────────────────────────────────
lam_vals   = list(results.keys())
err_interp = [results[l]['error_interp']  for l in lam_vals]
err_extrap = [results[l]['error_extrap']  for l in lam_vals]
phys_res   = [results[l]['phys_residual'] for l in lam_vals]

lam_plot = [max(l, 1e-4) for l in lam_vals]   # avoid log(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(lam_plot, err_interp, 'o-', lw=2, ms=8, color='steelblue')
axes[0].set_xscale('log')
axes[0].set_xlabel('λ (log scale)'); axes[0].set_ylabel('Mean absolute error')
axes[0].set_title('Interpolation error\n(inside training window [0, 2π])',
                  fontsize=11, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(lam_plot, err_extrap, 's-', lw=2, ms=8, color='tomato')
axes[1].set_xscale('log')
axes[1].set_xlabel('λ (log scale)'); axes[1].set_ylabel('Mean absolute error')
axes[1].set_title('Extrapolation error ★\n(outside training window [2π, 4π])',
                  fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3)

axes[2].plot(lam_plot, phys_res, '^-', lw=2, ms=8, color='seagreen')
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('λ (log scale)'); axes[2].set_ylabel('|u\"\" + u|')
axes[2].set_title('ODE residual\n(how well physics is satisfied)',
                  fontsize=11, fontweight='bold')
axes[2].grid(alpha=0.3, which='both')

plt.suptitle('Effect of λ on three key metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Reading the sweep results

### Why does an optimal λ exist at all?

The total loss is $\mathcal{L} = \mathcal{L}_{\text{data}} + \lambda\,\mathcal{L}_{\text{physics}}$. These two terms are in tension:

- **$\mathcal{L}_{\text{data}}$** wants the network to pass through the 5 training points. It is indifferent to what happens outside the training window — it would be perfectly happy with any wild extrapolation as long as the in-sample residual is small.
- **$\mathcal{L}_{\text{physics}}$** wants the network to satisfy $u'' + u = 0$ everywhere along the collocation grid. It is indifferent to the data — it would accept any function that solves the ODE, including those far from the measurements.

The optimal $\lambda$ is the one at which these two constraints cooperate rather than fight. As $\lambda$ moves away from that sweet spot in either direction, the model must sacrifice one objective to satisfy the other:

| λ | Data loss | Physics loss | Extrapolation |
|---|---|---|---|
| Too small | Low (fits data) | Large (physics ignored) | Poor — NN has no guidance |
| Optimal | Low (fits data) | Low (ODE satisfied) | Good — both terms agree |
| Too large | Moderate (physics dominates) | Low | Poor — network abandons the data |

### What to look for in the plots

1. **Extrapolation error** (centre panel, marked ★) is the decisive metric. It should dip to a minimum somewhere in the middle of the log-scale axis. Read off the $\lambda$ at that dip — that is your operating point.

2. **Interpolation error** (left panel) is less informative. Because the training window and the ODE constraint are compatible (the exact answer is $\sin x$), this tends to stay low across the whole sweep. Do not use it alone to select $\lambda$.

3. **Physics residual** (right panel) decreases monotonically. This is expected — you are literally paying more for physics enforcement as $\lambda$ grows. But a very low residual at the cost of high extrapolation error (over-constrained regime) is not what you want.

### The practical problem

In this notebook we can compute the extrapolation error exactly because we *know* the true solution $\sin(x)$. In a real problem you will not have that. The next section shows how to find the optimum programmatically when the ground truth is available (as here, for benchmarking), and discusses what to do when it is not.

## 8. Finding the optimal λ programmatically

The coarse 6-point sweep above is enough to see the shape of the curve. To pin down $\lambda^*$ precisely we run a denser logarithmic grid and take the argmin of the extrapolation error.

The logic is exactly:

$$\lambda^* = \arg\min_{\lambda \in \Lambda}\; \text{MAE}_{\text{extrap}}(\lambda)$$

where $\Lambda$ is a finite grid of candidate values and $\text{MAE}_{\text{extrap}}$ is the mean absolute error (MAE) on the held-out extrapolation region $[2\pi, 4\pi]$.

**Note on the held-out set.** Here we can evaluate directly against $\sin(x)$ because the ground truth is known. In a real problem you cannot do this. Practical alternatives:

| Proxy | How | When it works |
|---|---|---|
| **Validation split** | Hold out a random fraction of the training data; use validation MAE to score each λ | When you have enough data to afford a split |
| **Physics residual on a test grid** | Use $\overline{|u'' + u|}$ on a dense grid as the score | When the ODE is the right equation and there is no model error |
| **Cross-validation (CV)** | k-fold CV over the training data | Small-data regime; expensive |

For this tutorial we use the exact solution — the equivalent of a perfect validation set — to keep the focus on the λ-selection logic rather than the cross-validation machinery.

In [ ]:
# 25 log-spaced values from λ=0.001 to λ=10  (≈2 min on CPU)
lambdas_dense = np.logspace(-3, 1, 25)

err_interp_d = np.zeros(len(lambdas_dense))
err_extrap_d = np.zeros(len(lambdas_dense))
phys_res_d   = np.zeros(len(lambdas_dense))

print(f"Dense sweep: {len(lambdas_dense)} values from {lambdas_dense[0]:.4f} to {lambdas_dense[-1]:.1f}")
for i, lam in enumerate(lambdas_dense):
    model_d, _ = train(lam, x_train_t, y_train_t, x_col, seed=0)
    with torch.no_grad():
        pred_d = model_d(x_eval_t).numpy()

    x_phys_d = x_eval_t.clone().requires_grad_(True)
    u_d   = model_d(x_phys_d)
    du_d  = torch.autograd.grad(u_d,  x_phys_d, torch.ones_like(u_d),  create_graph=True)[0]
    d2u_d = torch.autograd.grad(du_d, x_phys_d, torch.ones_like(du_d), create_graph=True)[0]
    res_d = (d2u_d + u_d).detach().numpy()

    err_interp_d[i] = np.mean(np.abs(pred_d[:200] - y_true[:200]))
    err_extrap_d[i] = np.mean(np.abs(pred_d[200:] - y_true[200:]))
    phys_res_d[i]   = np.mean(np.abs(res_d))

# ── Compute the optimal λ ─────────────────────────────────────────────────────
idx_opt = int(np.argmin(err_extrap_d))
lam_opt = lambdas_dense[idx_opt]

print(f"\n{'─'*50}")
print(f"  λ*  (argmin extrapolation MAE) = {lam_opt:.5f}")
print(f"  Extrapolation MAE at λ*        = {err_extrap_d[idx_opt]:.5f}")
print(f"  Interpolation MAE at λ*        = {err_interp_d[idx_opt]:.5f}")
print(f"  Physics residual  at λ*        = {phys_res_d[idx_opt]:.5f}")
print(f"{'─'*50}")
print(f"  Extrapolation MAE at λ=0       = {err_extrap_d[0]:.5f}  (no physics — baseline)")
print(f"  Extrapolation MAE at λ=10      = {err_extrap_d[-1]:.5f}  (physics dominant)")

# ── Plot the error curves with λ* marked ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(lambdas_dense, err_interp_d, 'o-', lw=2, ms=5, color='steelblue',
         label='Interpolation error  [0, 2π]')
ax1.plot(lambdas_dense, err_extrap_d, 's-', lw=2, ms=5, color='tomato',
         label='Extrapolation error  [2π, 4π] ★')
ax1.axvline(lam_opt, color='gold', lw=2.5, ls='--', zorder=3,
            label=f'λ* = {lam_opt:.4f}')
ax1.scatter([lam_opt], [err_extrap_d[idx_opt]], s=140, zorder=6,
            color='gold', edgecolors='k', lw=1.5)
ax1.set_xscale('log')
ax1.set_xlabel('λ  (log scale)', fontsize=11)
ax1.set_ylabel('Mean absolute error', fontsize=11)
ax1.set_title('MAE vs λ  —  minimum of extrapolation error gives λ*',
              fontsize=11, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2.plot(lambdas_dense, phys_res_d, '^-', lw=2, ms=5, color='seagreen',
         label='ODE residual  |u\'\' + u|')
ax2.axvline(lam_opt, color='gold', lw=2.5, ls='--', zorder=3,
            label=f'λ* = {lam_opt:.4f}')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('λ  (log scale)', fontsize=11)
ax2.set_ylabel('Mean |u\'\' + u|  (log scale)', fontsize=11)
ax2.set_title('Physics residual vs λ  —  monotone decreasing',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, which='both')

plt.suptitle(f'Dense λ sweep  —  optimal value  λ* = {lam_opt:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Show the optimal model's prediction ──────────────────────────────────────
model_opt, _ = train(lam_opt, x_train_t, y_train_t, x_col, seed=0)
with torch.no_grad():
    pred_opt = model_opt(x_eval_t).numpy()

model_base, _ = train(0.0, x_train_t, y_train_t, x_col, seed=0)
with torch.no_grad():
    pred_base = model_base(x_eval_t).numpy()

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, pred, col, title in [
    (axA, pred_base, 'tomato',    r'$\lambda = 0$  (no physics)'),
    (axB, pred_opt,  'royalblue', rf'$\lambda^* = {lam_opt:.4f}$  (optimal)'),
]:
    ax.plot(x_eval, y_true, 'k-', lw=2, alpha=0.7, label='Exact sin(x)')
    ax.plot(x_eval, pred,   '-',  lw=2, color=col,  label='Prediction')
    ax.scatter(x_train, y_train, s=60, c='green', zorder=5, label='Training data')
    ax.axvspan(TRAIN_LO, TRAIN_HI,  alpha=0.08, color='green')
    ax.axvspan(TRAIN_HI, EXTRAP_HI, alpha=0.06, color='red')
    ax.axvline(TRAIN_HI, color='gray', lw=1, ls=':')
    ax.set_title(title, fontsize=12, color=col, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('u(x)')
    ax.set_ylim(-3, 3); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('No physics vs optimal λ*', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Practical guidelines for choosing λ

### The procedure

1. **Coarse log-spaced grid search** — train for λ ∈ {0.001, 0.01, 0.1, 1, 10}. Five runs. Plot extrapolation error vs λ and identify the approximate region of the minimum.
2. **Fine search around the minimum** — narrow down with a denser grid in that region (e.g., 10–15 log-spaced values). Take `lam_opt = lambdas[np.argmin(err_extrap)]`. This is what the code above does in one pass.
3. **Check the loss balance at epoch 0** — print both $\mathcal{L}_{\text{data}}$ and $\lambda\,\mathcal{L}_{\text{physics}}$ before the first gradient step. If they differ by more than 2–3 orders of magnitude, consider rescaling λ so they start at the same order. A very unbalanced initialisation means one term dominates throughout training regardless of λ.
4. **Watch both losses during training** — if the data loss stagnates while the physics loss keeps falling, λ is too large. If the physics loss barely moves from its initial value, λ is too small.

### When you do not have a validation set

| Situation | What to do |
|---|---|
| **Enough data to split** | Hold out 20% of training points as a validation set; use validation mean absolute error (MAE) to score each λ. |
| **Very few data points** | Use the ODE residual on a dense collocation grid as a proxy. It is a biased proxy (always decreases with λ) so combine it with a sanity check on the training data fit. |
| **No data at all** | You are in the pure-PINN regime (like the spring PINN in this course). λ only controls the relative weight of IC / BC terms vs ODE residual — tune so that initial / boundary conditions are satisfied to the required precision. |

### Common starting values in the literature

| Problem type | Typical λ range |
|---|---|
| Simple ODEs (as here) | 0.01 – 0.5 |
| Heat / wave equation | 0.1 – 1 |
| Burgers' equation (near shock) | 1 – 10 |
| Inverse problems | 10 – 100 |

These are starting points, not rules. Always validate against a held-out set or the exact solution when available.

### When a single scalar λ is not enough

When the loss has multiple components — ODE residual, boundary conditions, initial conditions — a single scalar λ is a blunt instrument. More systematic strategies covered in advanced PINN literature include:
- **Adaptive loss balancing** via Neural Tangent Kernel (NTK) analysis
- **Per-component weight schedules** that evolve during training
- **Self-adaptive weights** that are learned as additional parameters alongside the network

## Exercises

1. **Leave-one-out cross-validation proxy**: instead of using the exact solution to select $\lambda^*$, implement leave-one-out cross-validation over the 5 training points. For each candidate $\lambda$, train on 4 points and evaluate on the held-out point; average over all 5 leave-one-out splits. Compare the $\lambda^*$ found this way with the exact-solution argmin.

2. **Different ODE**: repeat the entire sweep for $u'' - u = 0$ (with exact solution $u = \cosh(x)$ on $[0, 2]$, IC: $u(0) = 1$, $u'(0) = 0$). Does the optimal $\lambda^*$ shift relative to the harmonic oscillator case? Explain why in terms of the magnitude of the ODE residual.

3. **Physics residual as a proxy**: in the dense sweep, use the mean physics residual $\overline{|u'' + u|}$ as a proxy score instead of the extrapolation error. Does it identify the same $\lambda^*$? Under what conditions would this proxy fail (hint: consider $\lambda \to \infty$)?

4. **Two-weight search**: extend the training objective to $\mathcal{L} = \mathcal{L}_{\text{data}} + \lambda_{\text{phys}} \mathcal{L}_{\text{phys}} + \lambda_{\text{IC}} \mathcal{L}_{\text{IC}}$, where $\lambda_{\text{IC}}$ penalises the initial condition $u(0) = 1$ separately. Run a 2D grid search over $(\lambda_{\text{phys}}, \lambda_{\text{IC}}) \in \{0.01, 0.1, 1\}^2$ and visualise the extrapolation error as a heatmap.